Gemini Content Moderation

Install Dependencies
pip install google-genai pydantic

In [ ]:
!pip install google-genai pydantic

In [ ]:
import os
from enum import Enum
from google import genai
from google.genai import types
from pydantic import BaseModel, Field

In [ ]:
# Ensure your API key is configured

from google.colab import userdata

os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')

In [ ]:
class HarmCategoryEnum(str, Enum):
    SAFE = "Safe"
    TOXICITY = "Toxicity or Hate Speech"
    HARASSMENT = "Harassment or Cyberbullying"
    SEXUAL = "Sexually Explicit Content"
    VIOLENCE = "Violence or Dangerous Acts"
    PII = "Personally Identifiable Information"
    ERROR = "Moderation API Error"


In [ ]:
# 2. Define the exact JSON structure you want Gemini to output
class ModerationResult(BaseModel):
    flagged: bool = Field(
        description="True if the text violates community standards or falls into a harmful category."
    )
    primary_category: HarmCategoryEnum = Field(
        description="The primary harm category matched. Select 'Safe' if the content passes."
    )
    confidence_score: float = Field(
        description="Confidence score between 0.0 (low confidence) and 1.0 (absolute certainty)."
    )
    reasoning: str = Field(
        description="A brief, 1-sentence explanation of why the text was flagged or cleared."
    )

In [ ]:
import asyncio
async_client = genai.Client()

async def moderate_content_single(user_text: str, max_retries: int = 3, initial_delay: float = 1.0) -> ModerationResult:
    print(f"Moderating: {user_text}")

    retries = 0
    while retries < max_retries:
        try:
            # Define system instructions to give Gemini its persona and rules
            system_instruction = (
                "You are an enterprise content moderation system. Analyze the user text objectively. "
                "Ignore spelling attempts to bypass filters (e.g., symbol substitution). Do not moralize, "
                "simply classify the text according to the provided schema instructions."
            )

            # CRITICAL STEP: Turn off internal filters so Gemini can safely ingest the bad text to evaluate it.
            disable_internal_safety = [
                types.SafetySetting(
                    category=types.HarmCategory.HARM_CATEGORY_HATE_SPEECH,
                    threshold=types.HarmBlockThreshold.BLOCK_NONE,
                ),
                types.SafetySetting(
                    category=types.HarmCategory.HARM_CATEGORY_HARASSMENT,
                    threshold=types.HarmBlockThreshold.BLOCK_NONE,
                ),
                types.SafetySetting(
                    category=types.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT,
                    threshold=types.HarmBlockThreshold.BLOCK_NONE,
                ),
                types.SafetySetting(
                    category=types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
                    threshold=types.HarmBlockThreshold.BLOCK_NONE,
                ),
            ]

            # Call Gemini with the structured configuration
            response = await async_client.models.generate_content(
                model="gemini-3.5-flash-lite", # Use Flash for ultra-fast, cheap classification
                contents=f"Please moderate the following text:\n\n{user_text}",
                config=types.GenerateContentConfig(
                    system_instruction=system_instruction,
                    safety_settings=disable_internal_safety,
                    temperature=0.0, # Forces deterministic, consistent classifications
                    response_mime_type="application/json", # Tells Gemini to speak JSON
                    response_schema=ModerationResult,     # Enforces the Pydantic structural format
                    thinking_config=types.ThinkingConfig(thinking_budget=1), # Turn off thinking steps
                ),
            )

            # Automatically returns the output parsed directly into your Pydantic object
            return response.parsed
        except Exception as e:
            retries += 1
            print(f"Error moderating text (attempt {retries}/{max_retries}): '{user_text}' - {e}")
            if retries < max_retries:
                delay = initial_delay * (2 ** (retries - 1)) # Exponential backoff
                print(f"Retrying in {delay:.2f} seconds...")
                await asyncio.sleep(delay)
            else:
                print(f"Failed to moderate text after {max_retries} attempts: '{user_text}'")
                # Return a default ModerationResult indicating an error
                return ModerationResult(
                    flagged=True,
                    primary_category=HarmCategoryEnum.ERROR,
                    confidence_score=0.0,
                    reasoning=f"Failed to moderate content due to API error: {e}"
                )


# --- TEST THE MODERATOR ---

In [ ]:
test_prompts = [
    "Hey! I really love the project you built, do you have a github link?",
    "You are an idiot and nobody likes you, I hope you lose your job tomorrow.",
    "Can you help me write a Python script that scrapes headlines from a website?"
]

In [ ]:
print("--- Running Moderation Tests ---")
for text in test_prompts:
    result = moderate_content(text)
    print(f"\n[Input]: \"{text}\"")
    print(f" Flagged: {result.flagged}")
    print(f" Category: {result.primary_category.value}")
    print(f" Confidence: {result.confidence_score}")
    print(f" Reason: {result.reasoning}")

In [ ]:
async def moderate_batch(texts: list[str]):
    # Define a semaphore to limit concurrent API requests
    # For the free tier (15 requests/minute), a concurrency limit of 3 is conservative.
    concurrency_limit = 3
    semaphore = asyncio.Semaphore(concurrency_limit)

    async def _moderate_single_with_semaphore(text: str):
        async with semaphore:
            return await moderate_content_single(text)

    # Process all text prompts, respecting the concurrency limit
    tasks = [_moderate_single_with_semaphore(t) for t in texts]
    results = await asyncio.gather(*tasks)
    return results

In [ ]:
print(await moderate_batch(test_prompts))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
input_file = '/content/drive/MyDrive/content-moderation-dataset/Aegis-AI-Content-Safety-Dataset-2.0/test.json'
output_file = '/content/drive/MyDrive/content-moderation-dataset/Aegis-AI-Content-Safety-Dataset-2.0/test-out.json'

In [ ]:
import pandas as pd


In [ ]:
df = pd.read_json(input_file)
display(df.head())

### Moderating the DataFrame Content Asynchronously

I will now apply the `moderate_content_single` function to the `text` column of your DataFrame (`df`) asynchronously, store the results, and then save the entire DataFrame with the new moderation results to the `output_file` as a JSON file.

In [ ]:
import asyncio

# Using 'prompt' as the column containing the content to be moderated
# If your content is in a different column, please adjust this line
print(f"Starting asynchronous moderation for {len(df)} items...")

# Define a batch size to stay within API rate limits (e.g., 10-15 requests/minute for free tier)
batch_size = 10 # Adjust this based on your API quota and model
all_moderation_results = []

for i in range(0, len(df), batch_size):
    batch_prompts = df['prompt'].iloc[i:i + batch_size].to_list()
    print(f"Processing batch {i//batch_size + 1}/{(len(df) + batch_size - 1)//batch_size} with {len(batch_prompts)} items...")
    batch_results = await moderate_batch(batch_prompts)
    all_moderation_results.extend(batch_results)

    # The API specifically suggested a retry delay of ~54 seconds due to quota exhaustion.
    # We need to wait for this duration to avoid hitting the rate limit repeatedly.
    # This delay is applied *between* batches.
    await asyncio.sleep(54) # Wait 54 seconds between batches to respect API rate limits

# Add the moderation results as a new column to the DataFrame
df['moderation_result'] = all_moderation_results

print("Moderation complete. Displaying first 5 rows with results:")
display(df.head())


In [ ]:
# Save the updated DataFrame to the output JSON file
df.to_json(output_file, orient='records', indent=4)
print(f"DataFrame with moderation results saved to: {output_file}")
